# 31. 41特徴量モデルの仕様固定と推論
出典：FX (3).ipynb、元セルindex [68, 69]。保存出力は results/imported_fx3/。
研究履歴の原本です。Notebookの変数・価格CSV・学習済みファイルに依存します。
失敗した試行も保管しています。一括実行やAPI接続を開始する入口ではありません。
元コード内の指示・自動判定名は資料として保存しています。独立した検証済みの結論とは区別してください。


## 元セルindex 68
構文状態：valid


In [ ]:
# ============================================================
# CHAMPION FREEZE + PRODUCTION MODEL BUILD
#
# FINAL CHAMPION:
#   BASE_PLUS_REGIME
#
# Purpose:
#   1. Final Tournament resultを再確認
#   2. 41 featuresを固定
#   3. 2025 validationで選ばれたLive policyを再現
#   4. Threshold / Session / Sizing等をFREEZE
#   5. 全clean historical dataでProduction HGBを再学習
#   6. Production calibratorを作成
#   7. Model / Config / Feature listを保存
#   8. Live inference用artifactを準備
#
# IMPORTANT:
#   - NO feature search
#   - NO model fallback
#   - NO parameter retuning
#   - NO 2026-based threshold optimization
# ============================================================

import json
import hashlib
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


# ============================================================
# 0. CONSTANTS
# ============================================================

CHAMPION_NAME = "BASE_PLUS_REGIME"

EXPECTED_DECISION = (
    "FREEZE_BASE_PLUS_REGIME"
)

POLICY_TEST_YEAR = 2026
POLICY_VALIDATION_YEAR = 2025


# ============================================================
# 1. HARD PRECHECK
# ============================================================

REQUIRED_OBJECTS = [

    "bars",

    "BASE_FEATURES",

    "HGB_CONFIG",

    "BASE_COST",

    "FINAL_TOURNAMENT_DECISION",

    "FINAL_TOURNAMENT_RECOMMENDED_CHAMPION",

    "FINAL_TOURNAMENT_FEATURE_SETS",

    "FINAL_TOURNAMENT_ANNUAL",

    "make_final_tournament_features",

    "build_tournament_dataset",

    "make_split",

    "choose_calibration",

    "expanding_oof",

    "fit_hgb",

    "model_probability",

    "fit_calibrator",

    "prediction_frame",

    "choose_threshold_session",

    "select_trades",

    "choose_sizing",

    "raw_position_size",

    "session_mask",
]


missing_objects = [

    name
    for name in REQUIRED_OBJECTS

    if name not in globals()
]


if missing_objects:

    raise RuntimeError(

        "Final TournamentのNotebook状態が不足しています。\n"

        f"Missing objects:\n{missing_objects}\n\n"

        "Final Feature Tournamentセルを先に実行してください。"
    )


# ============================================================
# 2. CHAMPION DECISION MUST MATCH
# ============================================================

if (
    FINAL_TOURNAMENT_DECISION
    !=
    EXPECTED_DECISION
):

    raise RuntimeError(

        "Final Tournament decisionが一致しません。\n"

        f"Expected: {EXPECTED_DECISION}\n"

        f"Actual:   {FINAL_TOURNAMENT_DECISION}\n\n"

        "Champion Freezeを停止します。"
    )


if (
    FINAL_TOURNAMENT_RECOMMENDED_CHAMPION
    !=
    CHAMPION_NAME
):

    raise RuntimeError(

        "Recommended Championが一致しません。\n"

        f"Expected: {CHAMPION_NAME}\n"

        f"Actual:   "
        f"{FINAL_TOURNAMENT_RECOMMENDED_CHAMPION}"
    )


print("=" * 110)

print(
    "CHAMPION DECISION VERIFIED"
)

print("=" * 110)

print(
    "Champion:",
    CHAMPION_NAME
)

print(
    "Tournament decision:",
    FINAL_TOURNAMENT_DECISION
)


# ============================================================
# 3. FREEZE FEATURE LIST
# ============================================================

if (
    CHAMPION_NAME
    not in
    FINAL_TOURNAMENT_FEATURE_SETS
):

    raise RuntimeError(
        "Champion feature listがありません。"
    )


CHAMPION_FEATURES = list(

    FINAL_TOURNAMENT_FEATURE_SETS[
        CHAMPION_NAME
    ]
)


if len(
    CHAMPION_FEATURES
) != 41:

    raise RuntimeError(

        "Champion feature countが41ではありません。\n"

        f"Actual: {len(CHAMPION_FEATURES)}"
    )


if len(
    set(
        CHAMPION_FEATURES
    )
) != len(
    CHAMPION_FEATURES
):

    raise RuntimeError(
        "Champion featuresにduplicateがあります。"
    )


print()

print("=" * 110)

print(
    "FEATURE FREEZE"
)

print("=" * 110)

print(
    "Feature count:",
    len(
        CHAMPION_FEATURES
    )
)


for i, feature in enumerate(
    CHAMPION_FEATURES,
    start=1
):

    print(
        f"{i:02d}. {feature}"
    )


# ============================================================
# 4. CLEAN BARS
# ============================================================

B_PRODUCTION = bars.copy()


B_PRODUCTION.columns = [

    str(c)
    .strip()
    .lower()

    for c in B_PRODUCTION.columns
]


required_ohlc = [

    "open",
    "high",
    "low",
    "close",
]


missing_ohlc = [

    c
    for c in required_ohlc

    if c not in
    B_PRODUCTION.columns
]


if missing_ohlc:

    raise RuntimeError(
        f"OHLC missing: {missing_ohlc}"
    )


B_PRODUCTION = (

    B_PRODUCTION[
        required_ohlc
    ]

    .copy()
)


if not isinstance(
    B_PRODUCTION.index,
    pd.DatetimeIndex
):

    raise RuntimeError(
        "bars.index must be DatetimeIndex"
    )


B_PRODUCTION.index = pd.to_datetime(

    B_PRODUCTION.index,

    utc=True,

    errors="coerce"
)


B_PRODUCTION = (

    B_PRODUCTION

    .loc[
        ~B_PRODUCTION.index.isna()
    ]

    .sort_index()
)


if (
    B_PRODUCTION.index
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Duplicate timestamp found."
    )


bad_grid = (

    (
        B_PRODUCTION.index.minute
        %
        15
    )
    !=
    0

) | (

    B_PRODUCTION.index.second
    !=
    0

) | (

    B_PRODUCTION.index.microsecond
    !=
    0
)


if np.asarray(
    bad_grid
).any():

    raise RuntimeError(
        "Non-15m timestamp found."
    )


# ============================================================
# 5. REBUILD EXACT CHAMPION FEATURES
# ============================================================

ALL_FEATURE_FRAME_PRODUCTION = (

    make_final_tournament_features(
        B_PRODUCTION
    )
)


missing_features = [

    feature
    for feature
    in CHAMPION_FEATURES

    if feature not in
    ALL_FEATURE_FRAME_PRODUCTION.columns
]


if missing_features:

    raise RuntimeError(

        "Production feature construction failed.\n"

        f"Missing:\n{missing_features}"
    )


# build_tournament_dataset() references globals
OLD_B = globals().get(
    "B",
    None
)

OLD_ALL_FEATURE_FRAME = globals().get(
    "ALL_FEATURE_FRAME",
    None
)


globals()[
    "B"
] = B_PRODUCTION


globals()[
    "ALL_FEATURE_FRAME"
] = ALL_FEATURE_FRAME_PRODUCTION


try:

    CHAMPION_DATA = (

        build_tournament_dataset(
            CHAMPION_FEATURES
        )
    )

finally:

    # B / ALL_FEATURE_FRAME are restored later only
    # after all reused pipeline operations finish.
    pass


print()

print("=" * 110)

print(
    "CHAMPION DATASET"
)

print("=" * 110)

print(
    "Rows:",
    f"{len(CHAMPION_DATA):,}"
)

print(
    "Signal period:",
    CHAMPION_DATA.index.min(),
    "->",
    CHAMPION_DATA.index.max()
)

print(
    "Last label end:",
    CHAMPION_DATA[
        "label_end"
    ].max()
)


# ============================================================
# 6. RECONSTRUCT THE FROZEN LIVE POLICY
#
# IMPORTANT:
# Policy is selected using Validation 2025.
#
# We do NOT optimize threshold/session/sizing on 2026 results.
# ============================================================

ORIGINAL_BASE_FEATURES = list(
    BASE_FEATURES
)


try:

    globals()[
        "BASE_FEATURES"
    ] = list(
        CHAMPION_FEATURES
    )


    split = make_split(

        CHAMPION_DATA,

        POLICY_TEST_YEAR
    )


    if split is None:

        raise RuntimeError(
            "2026 production-policy split failed."
        )


    TRAIN_POLICY = split[
        "train"
    ]


    VALIDATION_POLICY = split[
        "validation"
    ]


    # --------------------------------------------------------
    # Calibration method selection
    # --------------------------------------------------------

    (
        FROZEN_CALIBRATION,

        CALIBRATION_SELECTION_TABLE

    ) = choose_calibration(

        TRAIN_POLICY,

        VALIDATION_POLICY
    )


    # --------------------------------------------------------
    # Validation predictions
    # --------------------------------------------------------

    train_oof = expanding_oof(
        TRAIN_POLICY
    )


    validation_model = fit_hgb(
        TRAIN_POLICY
    )


    validation_raw_probability = (

        model_probability(

            validation_model,

            VALIDATION_POLICY
        )
    )


    validation_calibrator = (

        fit_calibrator(

            FROZEN_CALIBRATION,

            train_oof
        )
    )


    validation_probability = np.clip(

        validation_calibrator.predict(
            validation_raw_probability
        ),

        0.0,

        1.0
    )


    validation_prediction = (

        prediction_frame(

            VALIDATION_POLICY,

            validation_probability
        )
    )


    # --------------------------------------------------------
    # Threshold + Session
    # --------------------------------------------------------

    (
        FROZEN_THRESHOLD,

        FROZEN_SESSION,

        THRESHOLD_SESSION_TABLE

    ) = choose_threshold_session(

        validation_prediction
    )


    validation_selected = select_trades(

        validation_prediction,

        FROZEN_THRESHOLD,

        FROZEN_SESSION
    )


    # --------------------------------------------------------
    # Position sizing
    # --------------------------------------------------------

    (
        FROZEN_SIZING,

        FROZEN_SIZING_SCALE,

        SIZING_SELECTION_TABLE

    ) = choose_sizing(

        validation_selected,

        FROZEN_THRESHOLD
    )


finally:

    globals()[
        "BASE_FEATURES"
    ] = ORIGINAL_BASE_FEATURES


# ============================================================
# 7. VERIFY AGAINST FINAL TOURNAMENT 2026 ROW
# ============================================================

reference_rows = (

    FINAL_TOURNAMENT_ANNUAL.loc[

        (
            FINAL_TOURNAMENT_ANNUAL[
                "feature_set"
            ]
            ==
            CHAMPION_NAME
        )

        &

        (
            FINAL_TOURNAMENT_ANNUAL[
                "test_year"
            ]
            ==
            POLICY_TEST_YEAR
        )

    ]

    .copy()
)


if len(
    reference_rows
) != 1:

    raise RuntimeError(

        "Final Tournamentの2026 Champion rowを"
        "一意に取得できません。"
    )


reference = (
    reference_rows.iloc[0]
)


policy_checks = {

    "calibration":
        str(
            reference[
                "calibration"
            ]
        )
        ==
        str(
            FROZEN_CALIBRATION
        ),

    "threshold":
        np.isclose(

            float(
                reference[
                    "threshold"
                ]
            ),

            float(
                FROZEN_THRESHOLD
            ),

            atol=1e-12
        ),

    "session":
        str(
            reference[
                "session"
            ]
        )
        ==
        str(
            FROZEN_SESSION
        ),

    "sizing":
        str(
            reference[
                "sizing"
            ]
        )
        ==
        str(
            FROZEN_SIZING
        ),
}


if not all(
    policy_checks.values()
):

    raise RuntimeError(

        "Frozen policy reproduction failed.\n"

        f"{policy_checks}\n\n"

        "Production buildを停止します。"
    )


print()

print("=" * 110)

print(
    "FROZEN LIVE POLICY"
)

print("=" * 110)

print(
    "Policy source validation year:",
    POLICY_VALIDATION_YEAR
)

print(
    "Calibration:",
    FROZEN_CALIBRATION
)

print(
    "Threshold:",
    FROZEN_THRESHOLD
)

print(
    "Session:",
    FROZEN_SESSION
)

print(
    "Sizing:",
    FROZEN_SIZING
)

print(
    "Sizing scale:",
    FROZEN_SIZING_SCALE
)

print(
    "Exit:",
    "FIXED_30_MINUTES"
)

print(
    "Overlap:",
    "PROHIBITED"
)

print(
    "Base cost:",
    BASE_COST
)

print()

print(
    "Policy reproduction:",
    policy_checks
)


# ============================================================
# 8. TRAIN FINAL PRODUCTION MODEL
#
# Architecture / features / policy are frozen.
#
# Model weights are now fitted using ALL labeled historical
# clean data available at freeze time.
# ============================================================

try:

    globals()[
        "BASE_FEATURES"
    ] = list(
        CHAMPION_FEATURES
    )


    print()

    print("=" * 110)

    print(
        "TRAINING PRODUCTION HGB"
    )

    print("=" * 110)


    PRODUCTION_MODEL = fit_hgb(
        CHAMPION_DATA
    )


    # --------------------------------------------------------
    # OOF predictions for calibration
    # --------------------------------------------------------

    print(
        "Building expanding OOF predictions..."
    )


    PRODUCTION_OOF = expanding_oof(
        CHAMPION_DATA
    )


    if len(
        PRODUCTION_OOF
    ) < 1000:

        raise RuntimeError(

            "Production calibration用OOF dataが"
            "少なすぎます。"
        )


    PRODUCTION_CALIBRATOR = (

        fit_calibrator(

            FROZEN_CALIBRATION,

            PRODUCTION_OOF
        )
    )


finally:

    globals()[
        "BASE_FEATURES"
    ] = ORIGINAL_BASE_FEATURES


print(
    "Production HGB trained."
)

print(
    "Training rows:",
    f"{len(CHAMPION_DATA):,}"
)

print(
    "Calibration OOF rows:",
    f"{len(PRODUCTION_OOF):,}"
)


# ============================================================
# 9. PORTABLE CALIBRATION ESTIMATOR
#
# Do not save notebook wrapper unnecessarily.
# ============================================================

if (
    FROZEN_CALIBRATION
    ==
    "RAW"
):

    PORTABLE_CALIBRATOR = None


elif hasattr(
    PRODUCTION_CALIBRATOR,
    "model"
):

    PORTABLE_CALIBRATOR = (
        PRODUCTION_CALIBRATOR.model
    )


else:

    raise RuntimeError(

        "Portable calibration estimatorを"
        "取得できません。"
    )


# ============================================================
# 10. DATA HASH
#
# Reproducibility check
# ============================================================

bars_hash_values = (

    pd.util
    .hash_pandas_object(

        B_PRODUCTION[
            required_ohlc
        ],

        index=True
    )

    .values
)


DATA_SHA256 = hashlib.sha256(

    bars_hash_values.tobytes()

).hexdigest()


FEATURE_SHA256 = hashlib.sha256(

    "\n".join(
        CHAMPION_FEATURES
    )
    .encode(
        "utf-8"
    )

).hexdigest()


# ============================================================
# 11. FREEZE DIRECTORY
# ============================================================

FREEZE_UTC = pd.Timestamp.now(
    tz="UTC"
)


VERSION_DATE = (
    FREEZE_UTC.strftime(
        "%Y%m%d"
    )
)


CHAMPION_VERSION = (

    f"champion_v1_"
    f"{CHAMPION_NAME.lower()}_"
    f"{VERSION_DATE}"
)


FREEZE_DIR = (

    Path(
        "production_champion"
    )

    /

    CHAMPION_VERSION
)


FREEZE_DIR.mkdir(

    parents=True,

    exist_ok=True
)


# ============================================================
# 12. SAVE PRODUCTION MODEL
# ============================================================

MODEL_PATH = (

    FREEZE_DIR

    /

    "hgb_model.joblib"
)


joblib.dump(

    PRODUCTION_MODEL,

    MODEL_PATH
)


# ============================================================
# 13. SAVE CALIBRATOR
# ============================================================

CALIBRATOR_PATH = None


if PORTABLE_CALIBRATOR is not None:

    CALIBRATOR_PATH = (

        FREEZE_DIR

        /

        "calibrator.joblib"
    )


    joblib.dump(

        PORTABLE_CALIBRATOR,

        CALIBRATOR_PATH
    )


# ============================================================
# 14. MANIFEST
# ============================================================

MANIFEST = {

    "champion_version":
        CHAMPION_VERSION,

    "freeze_time_utc":
        FREEZE_UTC.isoformat(),

    "champion_name":
        CHAMPION_NAME,

    "feature_count":
        len(
            CHAMPION_FEATURES
        ),

    "features":
        CHAMPION_FEATURES,

    "feature_sha256":
        FEATURE_SHA256,

    "model":
        "HistGradientBoostingClassifier",

    "model_config": {

        key:
            (
                value.item()
                if isinstance(
                    value,
                    np.generic
                )
                else value
            )

        for key, value
        in HGB_CONFIG.items()
    },

    "calibration":
        str(
            FROZEN_CALIBRATION
        ),

    "threshold":
        float(
            FROZEN_THRESHOLD
        ),

    "session":
        str(
            FROZEN_SESSION
        ),

    "sizing_policy":
        str(
            FROZEN_SIZING
        ),

    "sizing_scale":
        float(
            FROZEN_SIZING_SCALE
        ),

    "exit_policy":
        "FIXED_30_MINUTES",

    "entry_policy":
        "NEXT_15M_BAR_OPEN",

    "overlapping_positions":
        False,

    "cost_per_trade_return":
        float(
            BASE_COST
        ),

    "policy_selection_source":
        {
            "test_year":
                POLICY_TEST_YEAR,

            "validation_year":
                POLICY_VALIDATION_YEAR,
        },

    "training_rows":
        int(
            len(
                CHAMPION_DATA
            )
        ),

    "training_signal_start":
        CHAMPION_DATA.index.min().isoformat(),

    "training_signal_end":
        CHAMPION_DATA.index.max().isoformat(),

    "training_last_label_end":
        CHAMPION_DATA[
            "label_end"
        ].max().isoformat(),

    "clean_bars_rows":
        int(
            len(
                B_PRODUCTION
            )
        ),

    "clean_bars_start":
        B_PRODUCTION.index.min().isoformat(),

    "clean_bars_end":
        B_PRODUCTION.index.max().isoformat(),

    "clean_data_sha256":
        DATA_SHA256,

    "research_decision":
        str(
            FINAL_TOURNAMENT_DECISION
        ),

    "research_note":
        (
            "2026 was used as confirmation during research "
            "and is not treated as a globally pristine holdout. "
            "Future shadow/paper data is the next unseen validation."
        ),
}


MANIFEST_PATH = (

    FREEZE_DIR

    /

    "champion_manifest.json"
)


with open(

    MANIFEST_PATH,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        MANIFEST,

        f,

        ensure_ascii=False,

        indent=2
    )


# ============================================================
# 15. SAVE FEATURE LIST
# ============================================================

FEATURE_PATH = (

    FREEZE_DIR

    /

    "features.txt"
)


with open(

    FEATURE_PATH,

    "w",

    encoding="utf-8"

) as f:

    for feature in CHAMPION_FEATURES:

        f.write(
            feature
            +
            "\n"
        )


# ============================================================
# 16. SAVE POLICY SELECTION TABLES
# ============================================================

CALIBRATION_SELECTION_TABLE.to_csv(

    FREEZE_DIR
    /
    "calibration_selection_2025.csv",

    index=False
)


THRESHOLD_SESSION_TABLE.to_csv(

    FREEZE_DIR
    /
    "threshold_session_selection_2025.csv",

    index=False
)


SIZING_SELECTION_TABLE.to_csv(

    FREEZE_DIR
    /
    "sizing_selection_2025.csv",

    index=False
)


# ============================================================
# 17. DRY-RUN INFERENCE
#
# This is ONLY a system smoke test.
#
# The latest historical row has already been included in
# production training, so this is NOT performance evaluation.
# ============================================================

LIVE_FEATURE_CANDIDATES = (

    ALL_FEATURE_FRAME_PRODUCTION

    .dropna(
        subset=CHAMPION_FEATURES
    )

    .copy()
)


if len(
    LIVE_FEATURE_CANDIDATES
) == 0:

    raise RuntimeError(
        "Live feature candidateがありません。"
    )


latest_signal_time = (

    LIVE_FEATURE_CANDIDATES
    .index
    .max()
)


latest_row = (

    LIVE_FEATURE_CANDIDATES

    .loc[
        [
            latest_signal_time
        ],

        CHAMPION_FEATURES
    ]
)


raw_p_up = float(

    PRODUCTION_MODEL
    .predict_proba(
        latest_row
    )[
        0,
        1
    ]
)


if (
    FROZEN_CALIBRATION
    ==
    "RAW"
):

    calibrated_p_up = (
        raw_p_up
    )


elif (
    FROZEN_CALIBRATION
    ==
    "ISOTONIC"
):

    calibrated_p_up = float(

        PORTABLE_CALIBRATOR
        .predict(
            [
                raw_p_up
            ]
        )[
            0
        ]
    )


elif (
    FROZEN_CALIBRATION
    ==
    "PLATT"
):

    calibrated_p_up = float(

        PORTABLE_CALIBRATOR
        .predict_proba(

            np.array(
                [
                    [
                        raw_p_up
                    ]
                ]
            )

        )[
            0,
            1
        ]
    )


else:

    raise RuntimeError(
        "Unknown calibration method."
    )


calibrated_p_up = float(

    np.clip(

        calibrated_p_up,

        0.0,

        1.0
    )
)


confidence = max(

    calibrated_p_up,

    1.0
    -
    calibrated_p_up
)


side = (

    "BUY"

    if calibrated_p_up
    >=
    0.5

    else
    "SELL"
)


session_allowed = bool(

    session_mask(

        pd.DatetimeIndex(
            [
                latest_signal_time
            ]
        ),

        FROZEN_SESSION
    )[
        0
    ]
)


threshold_allowed = bool(

    confidence

    >=

    FROZEN_THRESHOLD
)


if (
    threshold_allowed
    and
    session_allowed
):

    action = side


    raw_size = float(

        raw_position_size(

            np.array(
                [
                    confidence
                ]
            ),

            FROZEN_THRESHOLD,

            FROZEN_SIZING
        )[
            0
        ]
    )


    position_size = float(

        np.clip(

            raw_size

            *

            FROZEN_SIZING_SCALE,

            0.25,

            2.0
        )
    )


else:

    action = (
        "NO_TRADE"
    )

    position_size = (
        0.0
    )


DRY_RUN_RESULT = {

    "signal_time":
        latest_signal_time,

    "raw_p_up":
        raw_p_up,

    "calibrated_p_up":
        calibrated_p_up,

    "confidence":
        confidence,

    "session_allowed":
        session_allowed,

    "threshold_allowed":
        threshold_allowed,

    "action":
        action,

    "position_size":
        position_size,
}


# ============================================================
# 18. RESTORE GLOBALS
# ============================================================

globals()[
    "BASE_FEATURES"
] = ORIGINAL_BASE_FEATURES


if OLD_B is not None:

    globals()[
        "B"
    ] = OLD_B


if OLD_ALL_FEATURE_FRAME is not None:

    globals()[
        "ALL_FEATURE_FRAME"
    ] = OLD_ALL_FEATURE_FRAME


# ============================================================
# 19. SAVE NOTEBOOK PRODUCTION VARIABLES
# ============================================================

PRODUCTION_CHAMPION_NAME = (
    CHAMPION_NAME
)

PRODUCTION_CHAMPION_VERSION = (
    CHAMPION_VERSION
)

PRODUCTION_FEATURES = list(
    CHAMPION_FEATURES
)

PRODUCTION_MODEL_OBJECT = (
    PRODUCTION_MODEL
)

PRODUCTION_CALIBRATOR_OBJECT = (
    PORTABLE_CALIBRATOR
)

PRODUCTION_CALIBRATION_METHOD = (
    FROZEN_CALIBRATION
)

PRODUCTION_THRESHOLD = float(
    FROZEN_THRESHOLD
)

PRODUCTION_SESSION = (
    FROZEN_SESSION
)

PRODUCTION_SIZING = (
    FROZEN_SIZING
)

PRODUCTION_SIZING_SCALE = float(
    FROZEN_SIZING_SCALE
)

PRODUCTION_MANIFEST = (
    MANIFEST.copy()
)

PRODUCTION_FREEZE_DIR = str(
    FREEZE_DIR
)

PRODUCTION_DRY_RUN = (
    DRY_RUN_RESULT.copy()
)


# ============================================================
# 20. FINAL OUTPUT
# ============================================================

print()

print("=" * 110)

print(
    "CHAMPION FREEZE COMPLETE"
)

print("=" * 110)


print(
    "Champion:",
    PRODUCTION_CHAMPION_NAME
)

print(
    "Version:",
    PRODUCTION_CHAMPION_VERSION
)

print(
    "Features:",
    len(
        PRODUCTION_FEATURES
    )
)

print(
    "Calibration:",
    PRODUCTION_CALIBRATION_METHOD
)

print(
    "Threshold:",
    PRODUCTION_THRESHOLD
)

print(
    "Session:",
    PRODUCTION_SESSION
)

print(
    "Sizing:",
    PRODUCTION_SIZING
)

print(
    "Sizing scale:",
    PRODUCTION_SIZING_SCALE
)

print(
    "Exit:",
    "30 minutes"
)

print(
    "Overlap:",
    "PROHIBITED"
)


print()

print(
    "Training rows:",
    f"{len(CHAMPION_DATA):,}"
)

print(
    "Clean data SHA256:",
    DATA_SHA256
)


print()

print(
    "Saved directory:"
)

print(
    FREEZE_DIR
)


print()

print(
    "Model:"
)

print(
    MODEL_PATH
)


print()

print(
    "Manifest:"
)

print(
    MANIFEST_PATH
)


if CALIBRATOR_PATH is not None:

    print()

    print(
        "Calibrator:"
    )

    print(
        CALIBRATOR_PATH
    )


print()

print("=" * 110)

print(
    "DRY-RUN INFERENCE"
)

print("=" * 110)


for key, value in (
    DRY_RUN_RESULT.items()
):

    print(
        f"{key}: {value}"
    )


print()

print("=" * 110)

print(
    "NEXT STEP"
)

print("=" * 110)

print(
    "Champion is now frozen."
)

print(
    "Do NOT perform additional feature/parameter optimization."
)

print(
    "Next: build the LIVE INFERENCE ENGINE that loads"
)

print(
    "this frozen model and processes one newly closed 15m bar."
)


## 元セルindex 69
構文状態：valid


In [ ]:
# ============================================================
# FROZEN CHAMPION - LIVE INFERENCE ENGINE v1
#
# Champion:
#   BASE_PLUS_REGIME
#
# Features:
#   41
#
# This cell:
#   1. Frozen production artifactsをdiskから読み込む
#   2. Manifest / Feature / Model整合性を検査
#   3. Live用41特徴量を独立計算
#   4. 最新の「確定済み15分足」から推論
#   5. BUY / SELL / NO_TRADE を返す
#   6. Position sizeを返す
#   7. Entry / Exit予定時刻を返す
#   8. Previous notebook dry-runとParity Check
#
# IMPORTANT:
#   - NO training
#   - NO retuning
#   - NO feature search
#   - NO model fallback
#   - NO calibrator fallback
# ============================================================

from pathlib import Path
import json
import math
import warnings

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 240)


# ============================================================
# 0. EXPECTED FROZEN SPEC
# ============================================================

EXPECTED_CHAMPION = "BASE_PLUS_REGIME"
EXPECTED_FEATURE_COUNT = 41

MIN_HISTORY_BARS = 150

REQUIRED_OHLC = [
    "open",
    "high",
    "low",
    "close",
]


# ============================================================
# 1. FIND FROZEN CHAMPION DIRECTORY
#
# まずNotebookのPRODUCTION_FREEZE_DIRを使う。
# 無ければproduction_championから自動探索。
# ============================================================

def find_frozen_champion_directory():

    # --------------------------------------------------------
    # Preferred:
    # Previous Freeze cell variable
    # --------------------------------------------------------

    if "PRODUCTION_FREEZE_DIR" in globals():

        candidate = Path(
            str(
                PRODUCTION_FREEZE_DIR
            )
        )

        if candidate.exists():

            return candidate


    # --------------------------------------------------------
    # Independent fallback:
    # only directory discovery, NOT model fallback
    # --------------------------------------------------------

    root = Path(
        "production_champion"
    )

    if not root.exists():

        raise RuntimeError(
            "production_champion フォルダが見つかりません。"
        )


    candidates = sorted(

        [
            p
            for p in root.iterdir()

            if (
                p.is_dir()

                and

                p.name.startswith(
                    "champion_v1_base_plus_regime_"
                )

                and

                (
                    p
                    /
                    "champion_manifest.json"
                ).exists()
            )
        ],

        key=lambda p: p.name
    )


    if not candidates:

        raise RuntimeError(
            "BASE_PLUS_REGIMEのFreeze artifactが見つかりません。"
        )


    # 最新version directory
    return candidates[-1]


LIVE_CHAMPION_DIR = (
    find_frozen_champion_directory()
)


# ============================================================
# 2. FILE PATHS
# ============================================================

MANIFEST_PATH = (
    LIVE_CHAMPION_DIR
    /
    "champion_manifest.json"
)

MODEL_PATH = (
    LIVE_CHAMPION_DIR
    /
    "hgb_model.joblib"
)

CALIBRATOR_PATH = (
    LIVE_CHAMPION_DIR
    /
    "calibrator.joblib"
)

FEATURE_PATH = (
    LIVE_CHAMPION_DIR
    /
    "features.txt"
)


required_files = [

    MANIFEST_PATH,
    MODEL_PATH,
    FEATURE_PATH,
]


for path in required_files:

    if not path.exists():

        raise RuntimeError(
            f"Required artifact missing: {path}"
        )


# ============================================================
# 3. LOAD MANIFEST
# ============================================================

with open(
    MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:

    LIVE_MANIFEST = json.load(
        f
    )


# ============================================================
# 4. LOAD FEATURE LIST FROM FILE
# ============================================================

with open(
    FEATURE_PATH,
    "r",
    encoding="utf-8"
) as f:

    LIVE_FEATURES_FROM_FILE = [

        line.strip()

        for line in f.readlines()

        if line.strip()
    ]


LIVE_FEATURES = list(

    LIVE_MANIFEST[
        "features"
    ]
)


# ============================================================
# 5. HARD ARTIFACT VALIDATION
# ============================================================

if (
    LIVE_MANIFEST[
        "champion_name"
    ]
    !=
    EXPECTED_CHAMPION
):

    raise RuntimeError(

        "Wrong Champion loaded.\n"

        f"Expected: {EXPECTED_CHAMPION}\n"

        f"Actual: "
        f"{LIVE_MANIFEST['champion_name']}"
    )


if (
    int(
        LIVE_MANIFEST[
            "feature_count"
        ]
    )
    !=
    EXPECTED_FEATURE_COUNT
):

    raise RuntimeError(

        "Wrong feature count.\n"

        f"Expected: {EXPECTED_FEATURE_COUNT}\n"

        f"Actual: "
        f"{LIVE_MANIFEST['feature_count']}"
    )


if (
    len(
        LIVE_FEATURES
    )
    !=
    EXPECTED_FEATURE_COUNT
):

    raise RuntimeError(
        "Manifest feature count mismatch."
    )


if (
    LIVE_FEATURES
    !=
    LIVE_FEATURES_FROM_FILE
):

    raise RuntimeError(

        "features.txt と Manifestの"
        "feature順序が一致しません。"
    )


if len(
    set(
        LIVE_FEATURES
    )
) != len(
    LIVE_FEATURES
):

    raise RuntimeError(
        "Duplicate feature names found."
    )


# ============================================================
# 6. LOAD MODEL
# ============================================================

LIVE_MODEL = joblib.load(
    MODEL_PATH
)


if not hasattr(
    LIVE_MODEL,
    "predict_proba"
):

    raise RuntimeError(
        "Loaded model does not support predict_proba()."
    )


if hasattr(
    LIVE_MODEL,
    "n_features_in_"
):

    if int(
        LIVE_MODEL.n_features_in_
    ) != EXPECTED_FEATURE_COUNT:

        raise RuntimeError(

            "Model input feature count mismatch.\n"

            f"Model: {LIVE_MODEL.n_features_in_}\n"

            f"Expected: {EXPECTED_FEATURE_COUNT}"
        )


# sklearnがfeature namesを保持している場合
if hasattr(
    LIVE_MODEL,
    "feature_names_in_"
):

    model_feature_names = list(
        LIVE_MODEL.feature_names_in_
    )

    if (
        model_feature_names
        !=
        LIVE_FEATURES
    ):

        raise RuntimeError(

            "Modelのfeature orderとManifestが"
            "一致しません。"
        )


# ============================================================
# 7. LOAD CALIBRATOR
# ============================================================

LIVE_CALIBRATION_METHOD = str(

    LIVE_MANIFEST[
        "calibration"
    ]
).upper()


if (
    LIVE_CALIBRATION_METHOD
    ==
    "RAW"
):

    LIVE_CALIBRATOR = None


else:

    if not CALIBRATOR_PATH.exists():

        raise RuntimeError(

            "Calibration method requires "
            "calibrator.joblib but it is missing."
        )


    LIVE_CALIBRATOR = joblib.load(
        CALIBRATOR_PATH
    )


# ============================================================
# 8. FROZEN POLICY
# ============================================================

LIVE_THRESHOLD = float(

    LIVE_MANIFEST[
        "threshold"
    ]
)


LIVE_SESSION = str(

    LIVE_MANIFEST[
        "session"
    ]
)


LIVE_SIZING_POLICY = str(

    LIVE_MANIFEST[
        "sizing_policy"
    ]
)


LIVE_SIZING_SCALE = float(

    LIVE_MANIFEST[
        "sizing_scale"
    ]
)


LIVE_BASE_COST = float(

    LIVE_MANIFEST[
        "cost_per_trade_return"
    ]
)


LIVE_OVERLAP_ALLOWED = bool(

    LIVE_MANIFEST[
        "overlapping_positions"
    ]
)


if LIVE_OVERLAP_ALLOWED:

    raise RuntimeError(

        "Frozen strategy unexpectedly allows "
        "overlapping positions."
    )


# ============================================================
# 9. EXPECTED FEATURE LIST
#
# これも独立検証する。
# ============================================================

EXPECTED_LIVE_FEATURES = [

    # --------------------------------------------------------
    # BASE 30
    # --------------------------------------------------------

    "return_1",
    "return_2",
    "return_4",
    "return_8",
    "return_16",

    "vol_4",
    "vol_8",
    "vol_16",
    "vol_32",

    "ma5_distance",
    "ma5_slope",

    "ma10_distance",
    "ma10_slope",

    "ma20_distance",
    "ma20_slope",

    "ma50_distance",
    "ma50_slope",

    "ma100_distance",
    "ma100_slope",

    "body",
    "upper_wick",
    "lower_wick",
    "range_pct",

    "rsi14",
    "atr14",

    "distance_high_16",
    "distance_low_16",

    "hour_sin",
    "hour_cos",

    "weekday",

    # --------------------------------------------------------
    # REGIME 11
    # --------------------------------------------------------

    "adx14",
    "adx28",

    "ma20_50_spread",
    "ma50_100_spread",

    "trend_strength_20",
    "trend_strength_50",

    "price_pos_20",
    "price_pos_50",

    "ma_alignment_score",
    "slope_alignment_score",

    "directional_persistence_16",
]


if (
    LIVE_FEATURES
    !=
    EXPECTED_LIVE_FEATURES
):

    raise RuntimeError(

        "Frozen feature list differs from the "
        "expected BASE_PLUS_REGIME 41-feature specification."
    )


# ============================================================
# 10. OHLC INPUT NORMALIZATION
# ============================================================

def normalize_live_bars(
    bars_input
):

    if not isinstance(
        bars_input,
        pd.DataFrame
    ):

        raise TypeError(
            "bars_input must be pandas DataFrame."
        )


    x = bars_input.copy()


    x.columns = [

        str(c)
        .strip()
        .lower()

        for c in x.columns
    ]


    missing = [

        c
        for c in REQUIRED_OHLC

        if c not in x.columns
    ]


    if missing:

        raise RuntimeError(
            f"Missing OHLC columns: {missing}"
        )


    x = x[
        REQUIRED_OHLC
    ].copy()


    if not isinstance(
        x.index,
        pd.DatetimeIndex
    ):

        raise RuntimeError(
            "bars_input.index must be DatetimeIndex."
        )


    x.index = pd.to_datetime(

        x.index,

        utc=True,

        errors="coerce"
    )


    if x.index.isna().any():

        raise RuntimeError(
            "Invalid timestamps found."
        )


    x = x.sort_index()


    if x.index.duplicated().any():

        duplicates = int(

            x.index
            .duplicated()
            .sum()
        )

        raise RuntimeError(

            f"Duplicate timestamps: {duplicates}"
        )


    # --------------------------------------------------------
    # exact 15-minute grid
    # --------------------------------------------------------

    bad_grid = (

        (
            x.index.minute
            %
            15
        )
        !=
        0

        |

        (
            x.index.second
            !=
            0
        )

        |

        (
            x.index.microsecond
            !=
            0
        )
    )


    if np.asarray(
        bad_grid
    ).any():

        raise RuntimeError(
            "Non-15m timestamps found."
        )


    # --------------------------------------------------------
    # numeric
    # --------------------------------------------------------

    for c in REQUIRED_OHLC:

        x[c] = pd.to_numeric(

            x[c],

            errors="coerce"
        )


    if x[
        REQUIRED_OHLC
    ].isna().any().any():

        raise RuntimeError(
            "OHLC contains NaN/non-numeric values."
        )


    if (

        x[
            REQUIRED_OHLC
        ]

        <=
        0

    ).any().any():

        raise RuntimeError(
            "OHLC contains non-positive prices."
        )


    # --------------------------------------------------------
    # OHLC relationship
    # --------------------------------------------------------

    bad_high = (

        x[
            "high"
        ]

        <

        x[
            [
                "open",
                "low",
                "close",
            ]
        ].max(
            axis=1
        )
    )


    bad_low = (

        x[
            "low"
        ]

        >

        x[
            [
                "open",
                "high",
                "close",
            ]
        ].min(
            axis=1
        )
    )


    if (
        bad_high.any()
        or
        bad_low.any()
    ):

        raise RuntimeError(
            "Invalid OHLC relationship found."
        )


    if len(
        x
    ) < MIN_HISTORY_BARS:

        raise RuntimeError(

            "Not enough historical bars.\n"

            f"Need at least: {MIN_HISTORY_BARS}\n"

            f"Received: {len(x)}"
        )


    return x


# ============================================================
# 11. RSI
#
# Frozen research definition:
# Simple rolling mean version.
# ============================================================

def live_rsi(
    close,
    period=14
):

    delta = (
        close.diff()
    )


    gain = delta.clip(
        lower=0
    )


    loss = (
        -delta.clip(
            upper=0
        )
    )


    avg_gain = (

        gain
        .rolling(
            period
        )
        .mean()
    )


    avg_loss = (

        loss
        .rolling(
            period
        )
        .mean()
    )


    rs = (

        avg_gain

        /

        avg_loss.replace(
            0,
            np.nan
        )
    )


    return (

        100

        -

        (
            100
            /
            (
                1
                +
                rs
            )
        )
    )


# ============================================================
# 12. TRUE RANGE
# ============================================================

def live_true_range(
    x
):

    previous_close = (

        x[
            "close"
        ]
        .shift(
            1
        )
    )


    return pd.concat(

        [

            x[
                "high"
            ]
            -
            x[
                "low"
            ],

            (
                x[
                    "high"
                ]
                -
                previous_close
            ).abs(),

            (
                x[
                    "low"
                ]
                -
                previous_close
            ).abs(),
        ],

        axis=1

    ).max(
        axis=1
    )


# ============================================================
# 13. ADX
#
# Exact research implementation.
# ============================================================

def live_adx(
    x,
    period=14
):

    high = x[
        "high"
    ]

    low = x[
        "low"
    ]


    up_move = (
        high.diff()
    )


    down_move = (
        -low.diff()
    )


    plus_dm = pd.Series(

        np.where(

            (
                up_move
                >
                down_move
            )

            &

            (
                up_move
                >
                0
            ),

            up_move,

            0.0
        ),

        index=x.index
    )


    minus_dm = pd.Series(

        np.where(

            (
                down_move
                >
                up_move
            )

            &

            (
                down_move
                >
                0
            ),

            down_move,

            0.0
        ),

        index=x.index
    )


    tr = live_true_range(
        x
    )


    tr_sum = (

        tr
        .rolling(
            period
        )
        .sum()
        .replace(
            0,
            np.nan
        )
    )


    plus_di = (

        100.0

        *

        plus_dm
        .rolling(
            period
        )
        .sum()

        /

        tr_sum
    )


    minus_di = (

        100.0

        *

        minus_dm
        .rolling(
            period
        )
        .sum()

        /

        tr_sum
    )


    denominator = (

        plus_di
        +
        minus_di

    ).replace(
        0,
        np.nan
    )


    dx = (

        100.0

        *

        (
            plus_di
            -
            minus_di
        ).abs()

        /

        denominator
    )


    return (

        dx
        .rolling(
            period
        )
        .mean()

        /

        100.0
    )


# ============================================================
# 14. LIVE FEATURE ENGINEERING
#
# IMPORTANT:
# Only current and past bars are used.
# No shift(-1), no future price.
# ============================================================

def make_live_features(
    bars_input
):

    x = normalize_live_bars(
        bars_input
    )


    # ========================================================
    # BASE FEATURES
    # ========================================================

    for n in [
        1,
        2,
        4,
        8,
        16,
    ]:

        x[
            f"return_{n}"
        ] = (

            x[
                "close"
            ]
            .pct_change(
                n
            )
        )


    for n in [
        4,
        8,
        16,
        32,
    ]:

        x[
            f"vol_{n}"
        ] = (

            x[
                "return_1"
            ]

            .rolling(
                n
            )

            .std()
        )


    # --------------------------------------------------------
    # Moving averages
    # --------------------------------------------------------

    MA = {}


    for p in [
        5,
        10,
        20,
        50,
        100,
    ]:

        ma = (

            x[
                "close"
            ]

            .rolling(
                p
            )

            .mean()
        )


        MA[
            p
        ] = ma


        x[
            f"ma{p}_distance"
        ] = (

            x[
                "close"
            ]

            /

            ma

            -

            1
        )


        x[
            f"ma{p}_slope"
        ] = (
            ma.pct_change()
        )


    # --------------------------------------------------------
    # Candle structure
    # --------------------------------------------------------

    candle_range = (

        x[
            "high"
        ]

        -

        x[
            "low"
        ]

    ).replace(
        0,
        np.nan
    )


    x[
        "body"
    ] = (

        x[
            "close"
        ]

        -

        x[
            "open"
        ]

    ) / candle_range


    x[
        "upper_wick"
    ] = (

        x[
            "high"
        ]

        -

        x[
            [
                "open",
                "close",
            ]
        ].max(
            axis=1
        )

    ) / candle_range


    x[
        "lower_wick"
    ] = (

        x[
            [
                "open",
                "close",
            ]
        ].min(
            axis=1
        )

        -

        x[
            "low"
        ]

    ) / candle_range


    x[
        "range_pct"
    ] = (

        x[
            "high"
        ]

        -

        x[
            "low"
        ]

    ) / x[
        "close"
    ]


    # --------------------------------------------------------
    # RSI
    # --------------------------------------------------------

    x[
        "rsi14"
    ] = (

        live_rsi(

            x[
                "close"
            ],

            14
        )

        /

        100.0
    )


    # --------------------------------------------------------
    # ATR
    # --------------------------------------------------------

    true_range = live_true_range(
        x
    )


    atr14_absolute = (

        true_range

        .rolling(
            14
        )

        .mean()
    )


    x[
        "atr14"
    ] = (

        atr14_absolute

        /

        x[
            "close"
        ]
    )


    # --------------------------------------------------------
    # Distance to rolling high / low
    # --------------------------------------------------------

    high16 = (

        x[
            "high"
        ]

        .rolling(
            16
        )

        .max()
    )


    low16 = (

        x[
            "low"
        ]

        .rolling(
            16
        )

        .min()
    )


    x[
        "distance_high_16"
    ] = (

        high16

        -

        x[
            "close"
        ]

    ) / x[
        "close"
    ]


    x[
        "distance_low_16"
    ] = (

        x[
            "close"
        ]

        -

        low16

    ) / x[
        "close"
    ]


    # --------------------------------------------------------
    # UTC time features
    # --------------------------------------------------------

    hour = (

        x.index.hour

        +

        (
            x.index.minute
            /
            60.0
        )
    )


    x[
        "hour_sin"
    ] = np.sin(

        2

        *

        np.pi

        *

        hour

        /

        24.0
    )


    x[
        "hour_cos"
    ] = np.cos(

        2

        *

        np.pi

        *

        hour

        /

        24.0
    )


    x[
        "weekday"
    ] = (

        x.index.dayofweek

        /

        4.0
    )


    # ========================================================
    # REGIME FEATURES
    # ========================================================

    x[
        "adx14"
    ] = live_adx(
        x,
        14
    )


    x[
        "adx28"
    ] = live_adx(
        x,
        28
    )


    ma20 = MA[
        20
    ]

    ma50 = MA[
        50
    ]

    ma100 = MA[
        100
    ]


    x[
        "ma20_50_spread"
    ] = (

        ma20

        /

        ma50

        -

        1
    )


    x[
        "ma50_100_spread"
    ] = (

        ma50

        /

        ma100

        -

        1
    )


    atr_safe = (

        atr14_absolute

        .replace(
            0,
            np.nan
        )
    )


    x[
        "trend_strength_20"
    ] = (

        (
            x[
                "close"
            ]

            -

            ma20
        ).abs()

        /

        atr_safe
    )


    x[
        "trend_strength_50"
    ] = (

        (
            x[
                "close"
            ]

            -

            ma50
        ).abs()

        /

        atr_safe
    )


    high20 = (

        x[
            "high"
        ]

        .rolling(
            20
        )

        .max()
    )


    low20 = (

        x[
            "low"
        ]

        .rolling(
            20
        )

        .min()
    )


    high50 = (

        x[
            "high"
        ]

        .rolling(
            50
        )

        .max()
    )


    low50 = (

        x[
            "low"
        ]

        .rolling(
            50
        )

        .min()
    )


    x[
        "price_pos_20"
    ] = (

        (
            x[
                "close"
            ]

            -

            low20
        )

        /

        (
            high20
            -
            low20
        ).replace(
            0,
            np.nan
        )

        -

        0.5
    )


    x[
        "price_pos_50"
    ] = (

        (
            x[
                "close"
            ]

            -

            low50
        )

        /

        (
            high50
            -
            low50
        ).replace(
            0,
            np.nan
        )

        -

        0.5
    )


    x[
        "ma_alignment_score"
    ] = (

        (
            ma20
            >
            ma50
        ).astype(
            float
        )

        +

        (
            ma50
            >
            ma100
        ).astype(
            float
        )

    ) / 2.0


    slope20 = (
        ma20.pct_change()
    )

    slope50 = (
        ma50.pct_change()
    )

    slope100 = (
        ma100.pct_change()
    )


    x[
        "slope_alignment_score"
    ] = (

        np.sign(
            slope20
        )

        +

        np.sign(
            slope50
        )

        +

        np.sign(
            slope100
        )

    ) / 3.0


    x[
        "directional_persistence_16"
    ] = (

        np.sign(

            x[
                "return_1"
            ]
        )

        .rolling(
            16
        )

        .mean()

        .abs()
    )


    x = x.replace(

        [
            np.inf,
            -np.inf,
        ],

        np.nan
    )


    return x


# ============================================================
# 15. CALIBRATION
# ============================================================

def live_calibrate_probability(
    raw_p_up
):

    raw_p_up = float(
        raw_p_up
    )


    if (
        LIVE_CALIBRATION_METHOD
        ==
        "RAW"
    ):

        calibrated = (
            raw_p_up
        )


    elif (
        LIVE_CALIBRATION_METHOD
        ==
        "ISOTONIC"
    ):

        if not hasattr(
            LIVE_CALIBRATOR,
            "predict"
        ):

            raise RuntimeError(
                "Isotonic calibrator invalid."
            )


        calibrated = float(

            LIVE_CALIBRATOR
            .predict(
                [
                    raw_p_up
                ]
            )[
                0
            ]
        )


    elif (
        LIVE_CALIBRATION_METHOD
        ==
        "PLATT"
    ):

        if not hasattr(
            LIVE_CALIBRATOR,
            "predict_proba"
        ):

            raise RuntimeError(
                "Platt calibrator invalid."
            )


        calibrated = float(

            LIVE_CALIBRATOR

            .predict_proba(

                np.array(

                    [
                        [
                            raw_p_up
                        ]
                    ]
                )

            )[
                0,
                1
            ]
        )


    else:

        raise RuntimeError(

            "Unknown calibration method: "
            f"{LIVE_CALIBRATION_METHOD}"
        )


    return float(

        np.clip(

            calibrated,

            0.0,

            1.0
        )
    )


# ============================================================
# 16. SESSION FILTER
#
# Kept generic even though frozen Champion = ALL.
# ============================================================

def live_session_allowed(
    signal_time
):

    signal_time = pd.Timestamp(
        signal_time
    )


    if signal_time.tzinfo is None:

        signal_time = (
            signal_time
            .tz_localize(
                "UTC"
            )
        )

    else:

        signal_time = (
            signal_time
            .tz_convert(
                "UTC"
            )
        )


    hour = int(
        signal_time.hour
    )


    if (
        LIVE_SESSION
        ==
        "ALL"
    ):

        return True


    if (
        LIVE_SESSION
        ==
        "UTC_13_24"
    ):

        return (
            hour
            >=
            13
        )


    if (
        LIVE_SESSION
        ==
        "UTC_21_24"
    ):

        return (
            hour
            >=
            21
        )


    if (
        LIVE_SESSION
        ==
        "EXCLUDE_08_13"
    ):

        return not (

            8
            <=
            hour
            <
            13
        )


    raise RuntimeError(

        "Unknown session policy: "
        f"{LIVE_SESSION}"
    )


# ============================================================
# 17. POSITION SIZING
# ============================================================

def live_position_size(
    confidence
):

    confidence = float(
        confidence
    )


    edge = (

        confidence

        -

        LIVE_THRESHOLD

    ) / max(

        1.0

        -

        LIVE_THRESHOLD,

        1e-12
    )


    edge = float(

        np.clip(

            edge,

            0.0,

            1.0
        )
    )


    if (
        LIVE_SIZING_POLICY
        ==
        "FIXED"
    ):

        raw_size = 1.0


    elif (
        LIVE_SIZING_POLICY
        ==
        "GENTLE"
    ):

        raw_size = (

            0.85

            +

            0.30
            *
            edge
        )


    elif (
        LIVE_SIZING_POLICY
        ==
        "MODERATE"
    ):

        raw_size = (

            0.70

            +

            0.60
            *
            edge
        )


    elif (
        LIVE_SIZING_POLICY
        ==
        "STRONG"
    ):

        raw_size = (

            0.50

            +

            1.00
            *
            edge
        )


    else:

        raise RuntimeError(

            "Unknown sizing policy: "
            f"{LIVE_SIZING_POLICY}"
        )


    final_size = (

        raw_size

        *

        LIVE_SIZING_SCALE
    )


    return float(

        np.clip(

            final_size,

            0.25,

            2.0
        )
    )


# ============================================================
# 18. MAIN LIVE INFERENCE FUNCTION
#
# Parameters:
#
# bars_input
#   最新確定足まで含むOHLC DataFrame
#
# position_is_open
#   TrueならOverlap禁止のため新規エントリーしない
#
# as_of_utc
#   実運用時に現在時刻を渡すことで、
#   最新barが本当に確定済みか確認可能。
#
# Example:
#
# result = run_live_inference(
#     latest_bars,
#     position_is_open=False,
#     as_of_utc=pd.Timestamp.now(tz="UTC")
# )
# ============================================================

def run_live_inference(
    bars_input,
    position_is_open=False,
    as_of_utc=None
):

    # --------------------------------------------------------
    # Normalize data
    # --------------------------------------------------------

    clean_bars = normalize_live_bars(
        bars_input
    )


    signal_time = (
        clean_bars.index[-1]
    )


    # --------------------------------------------------------
    # Optional closed-bar verification
    #
    # Timestamp denotes BAR OPEN TIME.
    # 00:00 bar closes at 00:15.
    # --------------------------------------------------------

    signal_bar_close_time = (

        signal_time

        +

        pd.Timedelta(
            minutes=15
        )
    )


    if as_of_utc is not None:

        as_of_utc = pd.Timestamp(
            as_of_utc
        )


        if as_of_utc.tzinfo is None:

            as_of_utc = (
                as_of_utc
                .tz_localize(
                    "UTC"
                )
            )

        else:

            as_of_utc = (
                as_of_utc
                .tz_convert(
                    "UTC"
                )
            )


        if (

            signal_bar_close_time

            >

            as_of_utc
        ):

            raise RuntimeError(

                "Latest 15m bar is not closed yet.\n"

                f"Bar open : {signal_time}\n"

                f"Bar close: {signal_bar_close_time}\n"

                f"Now      : {as_of_utc}"
            )


    # --------------------------------------------------------
    # Feature engineering
    # --------------------------------------------------------

    feature_frame = make_live_features(
        clean_bars
    )


    latest = (

        feature_frame

        .loc[
            [
                signal_time
            ],

            LIVE_FEATURES
        ]

        .copy()
    )


    # --------------------------------------------------------
    # NaN check
    # --------------------------------------------------------

    if latest.isna().any().any():

        missing_live_features = (

            latest.columns[
                latest.isna()
                .iloc[0]
            ]
            .tolist()
        )


        raise RuntimeError(

            "Latest signal has NaN features.\n"

            f"Missing features: {missing_live_features}"
        )


    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    raw_p_up = float(

        LIVE_MODEL

        .predict_proba(
            latest
        )[
            0,
            1
        ]
    )


    # --------------------------------------------------------
    # Calibration
    # --------------------------------------------------------

    calibrated_p_up = (

        live_calibrate_probability(
            raw_p_up
        )
    )


    # --------------------------------------------------------
    # Direction + confidence
    # --------------------------------------------------------

    if (
        calibrated_p_up
        >=
        0.5
    ):

        proposed_side = (
            "BUY"
        )

    else:

        proposed_side = (
            "SELL"
        )


    confidence = float(

        max(

            calibrated_p_up,

            1.0
            -
            calibrated_p_up
        )
    )


    # --------------------------------------------------------
    # Filters
    # --------------------------------------------------------

    session_allowed = (

        live_session_allowed(
            signal_time
        )
    )


    threshold_allowed = bool(

        confidence

        >=

        LIVE_THRESHOLD
    )


    # --------------------------------------------------------
    # Decision
    # --------------------------------------------------------

    if not session_allowed:

        action = (
            "NO_TRADE"
        )

        reason = (
            "SESSION_BLOCKED"
        )

        position_size = 0.0


    elif not threshold_allowed:

        action = (
            "NO_TRADE"
        )

        reason = (
            "LOW_CONFIDENCE"
        )

        position_size = 0.0


    elif bool(
        position_is_open
    ):

        action = (
            "NO_TRADE"
        )

        reason = (
            "OPEN_POSITION_OVERLAP_BLOCKED"
        )

        position_size = 0.0


    else:

        action = (
            proposed_side
        )

        reason = (
            "TRADE_SIGNAL"
        )

        position_size = (

            live_position_size(
                confidence
            )
        )


    # --------------------------------------------------------
    # Trade schedule
    #
    # signal_time      = t
    # bar closes       = t + 15m
    # next bar open    = t + 15m
    # hold 30 minutes
    # exit             = t + 45m
    # --------------------------------------------------------

    if (
        action
        in
        [
            "BUY",
            "SELL"
        ]
    ):

        planned_entry_time = (

            signal_time

            +

            pd.Timedelta(
                minutes=15
            )
        )


        planned_exit_time = (

            planned_entry_time

            +

            pd.Timedelta(
                minutes=30
            )
        )


    else:

        planned_entry_time = (
            pd.NaT
        )

        planned_exit_time = (
            pd.NaT
        )


    result = {

        "champion_version":
            LIVE_MANIFEST[
                "champion_version"
            ],

        "champion":
            LIVE_MANIFEST[
                "champion_name"
            ],

        "signal_time":
            signal_time,

        "signal_bar_close_time":
            signal_bar_close_time,

        "raw_p_up":
            raw_p_up,

        "calibrated_p_up":
            calibrated_p_up,

        "confidence":
            confidence,

        "proposed_side":
            proposed_side,

        "threshold":
            LIVE_THRESHOLD,

        "threshold_allowed":
            threshold_allowed,

        "session":
            LIVE_SESSION,

        "session_allowed":
            session_allowed,

        "position_was_open":
            bool(
                position_is_open
            ),

        "action":
            action,

        "reason":
            reason,

        "position_size":
            position_size,

        "planned_entry_time":
            planned_entry_time,

        "planned_exit_time":
            planned_exit_time,

        "cost_assumption":
            LIVE_BASE_COST,

        "feature_count":
            len(
                LIVE_FEATURES
            ),
    }


    return result


# ============================================================
# 19. HUMAN-READABLE RESULT
# ============================================================

def print_live_signal(
    result
):

    print()

    print("=" * 90)

    print(
        "FROZEN CHAMPION LIVE SIGNAL"
    )

    print("=" * 90)


    print(
        "Champion:",
        result[
            "champion"
        ]
    )


    print(
        "Version:",
        result[
            "champion_version"
        ]
    )


    print(
        "Signal bar:",
        result[
            "signal_time"
        ]
    )


    print(
        "Bar closed:",
        result[
            "signal_bar_close_time"
        ]
    )


    print()

    print(
        "Raw p_up:",
        f"{result['raw_p_up']:.8f}"
    )


    print(
        "Calibrated p_up:",
        f"{result['calibrated_p_up']:.8f}"
    )


    print(
        "Confidence:",
        f"{result['confidence']:.8f}"
    )


    print(
        "Threshold:",
        result[
            "threshold"
        ]
    )


    print()

    print(
        "Proposed side:",
        result[
            "proposed_side"
        ]
    )


    print(
        "Threshold allowed:",
        result[
            "threshold_allowed"
        ]
    )


    print(
        "Session allowed:",
        result[
            "session_allowed"
        ]
    )


    print(
        "Existing position:",
        result[
            "position_was_open"
        ]
    )


    print()

    print(
        "ACTION:",
        result[
            "action"
        ]
    )


    print(
        "Reason:",
        result[
            "reason"
        ]
    )


    print(
        "Position size:",
        result[
            "position_size"
        ]
    )


    if (
        result[
            "action"
        ]
        in
        [
            "BUY",
            "SELL"
        ]
    ):

        print()

        print(
            "Planned entry:",
            result[
                "planned_entry_time"
            ]
        )


        print(
            "Planned exit:",
            result[
                "planned_exit_time"
            ]
        )


# ============================================================
# 20. ARTIFACT SUMMARY
# ============================================================

print("=" * 100)

print(
    "LIVE ENGINE ARTIFACT CHECK"
)

print("=" * 100)


print(
    "Directory:",
    LIVE_CHAMPION_DIR
)


print(
    "Champion:",
    LIVE_MANIFEST[
        "champion_name"
    ]
)


print(
    "Version:",
    LIVE_MANIFEST[
        "champion_version"
    ]
)


print(
    "Feature count:",
    len(
        LIVE_FEATURES
    )
)


print(
    "Model:",
    type(
        LIVE_MODEL
    ).__name__
)


print(
    "Calibration:",
    LIVE_CALIBRATION_METHOD
)


print(
    "Threshold:",
    LIVE_THRESHOLD
)


print(
    "Session:",
    LIVE_SESSION
)


print(
    "Sizing:",
    LIVE_SIZING_POLICY
)


print(
    "Sizing scale:",
    LIVE_SIZING_SCALE
)


print(
    "Overlap:",
    "PROHIBITED"
)


# ============================================================
# 21. NOTEBOOK SMOKE TEST
#
# Existing clean historical `bars` があれば実行。
# ============================================================

LIVE_ENGINE_SMOKE_RESULT = None


if "bars" in globals():

    print()

    print("=" * 100)

    print(
        "LIVE ENGINE SMOKE TEST"
    )

    print("=" * 100)


    LIVE_ENGINE_SMOKE_RESULT = (

        run_live_inference(

            bars,

            position_is_open=False,

            as_of_utc=None
        )
    )


    print_live_signal(
        LIVE_ENGINE_SMOKE_RESULT
    )


else:

    print()

    print(
        "[INFO] `bars` not found. "
        "Live engine defined, but smoke test skipped."
    )


# ============================================================
# 22. PARITY CHECK AGAINST PREVIOUS FREEZE DRY RUN
#
# Same latest bar must produce the same prediction.
# ============================================================

LIVE_ENGINE_PARITY_OK = None


if (

    "PRODUCTION_DRY_RUN"
    in globals()

    and

    LIVE_ENGINE_SMOKE_RESULT
    is not None

):

    expected = (
        PRODUCTION_DRY_RUN
    )


    actual = (
        LIVE_ENGINE_SMOKE_RESULT
    )


    parity_checks = {

        "signal_time":

            pd.Timestamp(
                actual[
                    "signal_time"
                ]
            )

            ==

            pd.Timestamp(
                expected[
                    "signal_time"
                ]
            ),


        "raw_p_up":

            np.isclose(

                float(
                    actual[
                        "raw_p_up"
                    ]
                ),

                float(
                    expected[
                        "raw_p_up"
                    ]
                ),

                atol=1e-10,

                rtol=1e-10
            ),


        "calibrated_p_up":

            np.isclose(

                float(
                    actual[
                        "calibrated_p_up"
                    ]
                ),

                float(
                    expected[
                        "calibrated_p_up"
                    ]
                ),

                atol=1e-10,

                rtol=1e-10
            ),


        "confidence":

            np.isclose(

                float(
                    actual[
                        "confidence"
                    ]
                ),

                float(
                    expected[
                        "confidence"
                    ]
                ),

                atol=1e-10,

                rtol=1e-10
            ),


        "session_allowed":

            bool(
                actual[
                    "session_allowed"
                ]
            )

            ==

            bool(
                expected[
                    "session_allowed"
                ]
            ),


        "threshold_allowed":

            bool(
                actual[
                    "threshold_allowed"
                ]
            )

            ==

            bool(
                expected[
                    "threshold_allowed"
                ]
            ),


        "action":

            str(
                actual[
                    "action"
                ]
            )

            ==

            str(
                expected[
                    "action"
                ]
            ),


        "position_size":

            np.isclose(

                float(
                    actual[
                        "position_size"
                    ]
                ),

                float(
                    expected[
                        "position_size"
                    ]
                ),

                atol=1e-10,

                rtol=1e-10
            ),
    }


    LIVE_ENGINE_PARITY_OK = all(

        parity_checks.values()
    )


    print()

    print("=" * 100)

    print(
        "FREEZE DRY-RUN PARITY CHECK"
    )

    print("=" * 100)


    for key, value in (
        parity_checks.items()
    ):

        print(
            f"{key}: {value}"
        )


    print()

    print(
        "PARITY PASSED:",
        LIVE_ENGINE_PARITY_OK
    )


    if not LIVE_ENGINE_PARITY_OK:

        raise RuntimeError(

            "Live Engine does not reproduce "
            "the frozen production dry-run.\n"

            "STOP before Paper Trading."
        )


# ============================================================
# 23. SAVE ENGINE STATE IN NOTEBOOK
# ============================================================

LIVE_ENGINE_CHAMPION_DIR = str(
    LIVE_CHAMPION_DIR
)

LIVE_ENGINE_MANIFEST = (
    LIVE_MANIFEST.copy()
)

LIVE_ENGINE_FEATURES = list(
    LIVE_FEATURES
)

LIVE_ENGINE_MODEL = (
    LIVE_MODEL
)

LIVE_ENGINE_CALIBRATOR = (
    LIVE_CALIBRATOR
)


# ============================================================
# 24. FINAL STATUS
# ============================================================

print()

print("=" * 100)

print(
    "LIVE INFERENCE ENGINE READY"
)

print("=" * 100)


print(
    "Champion:",
    EXPECTED_CHAMPION
)


print(
    "Features:",
    EXPECTED_FEATURE_COUNT
)


print(
    "Frozen model loaded: YES"
)


print(
    "Frozen calibrator loaded:",
    (
        "YES"
        if LIVE_CALIBRATOR is not None
        else "RAW"
    )
)


if LIVE_ENGINE_PARITY_OK is True:

    print(
        "Previous Dry-run parity: PASS"
    )


elif LIVE_ENGINE_PARITY_OK is None:

    print(
        "Previous Dry-run parity: NOT TESTED"
    )


else:

    print(
        "Previous Dry-run parity: FAIL"
    )


print()

print(
    "No training was performed."
)

print(
    "No optimization was performed."
)

print(
    "No future prices were used in feature construction."
)

print()

print(
    "NEXT STEP:"
)

print(
    "Historical Replay Parity Test."
)

print(
    "We will feed historical 15m bars to this exact"
)

print(
    "Live Engine sequentially and verify that its"
)

print(
    "signals match the frozen backtest behavior."
)
